In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler
from imblearn.pipeline import Pipeline as ImbPipeline
import warnings
warnings.filterwarnings('ignore')

data = pd.read_parquet('data/prepared_data.parquet')
print(data.shape)
print(f"Fraud procentas: {data['Is Fraud?'].mean()*100:.3f}%")

## Train Test split

In [ ]:
data['date'] = pd.to_datetime(data[['Year', 'Month', 'Day']])
data_sorted = data.sort_values('date').reset_index(drop=True)

split_i = int(len(data_sorted) * 0.85)
train_val_data = data_sorted.iloc[:split_i].copy()
test_data      = data_sorted.iloc[split_i:].copy()

train_val_data.drop(columns=['date', 'Year', 'Month', 'Day'], inplace=True)
test_data.drop(columns=['date', 'Year', 'Month', 'Day'], inplace=True)

print(f"Train: {len(train_val_data)} eil | fraud: {train_val_data['Is Fraud?'].mean()*100:.3f}%")
print(f"Test:  {len(test_data)} eil | fraud: {test_data['Is Fraud?'].mean()*100:.3f}%")

## Validacijos aibe


In [ ]:
X_tv = train_val_data.drop(columns=['Is Fraud?'])
y_tv = train_val_data['Is Fraud?']

X_test = test_data.drop(columns=['Is Fraud?'])
y_test = test_data['Is Fraud?']

X_train, X_val, y_train, y_val = train_test_split(
    X_tv, y_tv,
    test_size=0.20,
    random_state=67,
    stratify=y_tv
)

print(f"Train: {len(X_train)} eil | fraud: {y_train.mean()*100:.3f}%")
print(f"Val: {len(X_val)} eil | fraud: {y_val.mean()*100:.3f}%")
print(f"Test: {len(X_test)} eil | fraud: {y_test.mean()*100:.3f}%")

In [ ]:
from sklearn.metrics import (
    f1_score, precision_score, recall_score,
    roc_auc_score, average_precision_score,
    classification_report, confusion_matrix,
    precision_recall_curve, roc_curve
)
import matplotlib.pyplot as plt

def get_predictions(model, X, threshold=0.5):
    y_proba = model.predict_proba(X)[:, 1]
    y_pred  = (y_proba >= threshold).astype(int)
    return y_proba, y_pred


def print_metrics(name, label, threshold, y, y_pred, y_proba):
    print("*************************************************************************")
    print(f"{name} | {label} | threshold={threshold:.3f}")
    print("*************************************************************************")
    print(classification_report(y, y_pred, target_names=['Ne-fraud', 'Fraud']))
    print(f"Precision: {precision_score(y, y_pred):.4f}")
    print(f"Recall: {recall_score(y, y_pred):.4f}")
    print(f"F1: {f1_score(y, y_pred):.4f}")
    print(f"ROC-AUC: {roc_auc_score(y, y_proba):.4f}")
    print(f"PR-AUC: {average_precision_score(y, y_proba):.4f}")
    print()
    print(f"Confusion matrix: {confusion_matrix(y, y_pred)}")


def plot_pr_curve(ax, y, y_pred, y_proba, threshold):
    prec, rec, _ = precision_recall_curve(y, y_proba)
    pr_auc = average_precision_score(y, y_proba)
    ax.plot(rec, prec, label=f'PR-AUC = {pr_auc:.4f}')
    ax.scatter(recall_score(y, y_pred), precision_score(y, y_pred),
               color='red', zorder=5, label=f'threshold={threshold:.3f}')
    ax.set_xlabel('Recall')
    ax.set_ylabel('Precision')
    ax.set_title('Precision-Recall')
    ax.legend()
    ax.grid(True)


def plot_roc_curve(ax, y, y_proba):
    fpr, tpr, _ = roc_curve(y, y_proba)
    roc_auc = roc_auc_score(y, y_proba)
    ax.plot(fpr, tpr, label=f'ROC-AUC = {roc_auc:.4f}')
    ax.set_xlabel('False Positive Rate')
    ax.set_ylabel('True Positive Rate (Recall)')
    ax.set_title('ROC Curve')
    ax.legend()
    ax.grid(True)


def plot_curves(name, label, y, y_pred, y_proba, threshold):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    plot_pr_curve(axes[0], y, y_pred, y_proba, threshold)
    plot_roc_curve(axes[1], y, y_proba)
    plt.suptitle(f'{name} | {label}')
    plt.tight_layout()
    plt.show()


def full_evaluate(name, model, X, y, threshold = 0.5, label=''):
    y_proba, y_pred = get_predictions(model, X, threshold)
    print_metrics(name, label, threshold, y, y_pred, y_proba)
    plot_curves(name, label, y, y_pred, y_proba, threshold)
    results = {
        'precision': precision_score(y, y_pred),
        'recall': recall_score(y, y_pred),
        'f1': f1_score(y, y_pred),
        'roc_auc': roc_auc_score(y, y_proba),
        'pr_auc': average_precision_score(y, y_proba),
    }
    return results

def cv_score(model, X, y, cv=5, label=''):
    skf = StratifiedKFold(n_splits=cv, shuffle=True, random_state=67)
    scores = cross_validate(
        model, X, y,
        cv=skf,
        scoring=['roc_auc', 'average_precision'],
        n_jobs=-1
    )
    print()
    print(f"CV-{cv} | {label}]")
    print(f"ROC-AUC: {scores['test_roc_auc'].mean():.4f}")
    print(f"std ROC-AUC: {scores['test_roc_auc'].std():.4f}")
    print(f"PR-AUC: {scores['test_average_precision'].mean():.4f}")
    print(f"std PR-AUC: {scores['test_average_precision'].std():.4f}")
    return scores

def balance_data(X, y, fraud_target_ratio):
    ratio = fraud_target_ratio / (1 - fraud_target_ratio)
    target_for_under = ratio / 2

    n_fraud = (y == 1).sum()
    n_majority_target = int(n_fraud / target_for_under)
    under_ratio = n_majority_target / (y == 0).sum()
    under_ratio = min(under_ratio, 1.0)

    print(f"=== Prieš balansavimą ===")
    print(f"  Fraud: {n_fraud}, Non-fraud: {(y==0).sum()}")
    print(f"  Fraud %: {y.mean()*100:.1f}%")
    print(f"  under_ratio: {under_ratio:.4f}, target_for_under: {target_for_under:.4f}, ratio: {ratio:.4f}")

    under = RandomUnderSampler(sampling_strategy={0: n_majority_target, 1: n_fraud}, random_state=67)
    X_u, y_u = under.fit_resample(X, y)

    print(f"\n=== Po undersampling ===")
    print(f"  Fraud: {(y_u==1).sum()}, Non-fraud: {(y_u==0).sum()}")
    print(f"  Fraud %: {y_u.mean()*100:.1f}%")

    over = SMOTE(sampling_strategy=ratio, random_state=67)
    X_bal, y_bal = over.fit_resample(X_u, y_u)

    print(f"\n=== Po SMOTE ===")
    print(f"  Fraud: {(y_bal==1).sum()}, Non-fraud: {(y_bal==0).sum()}")
    print(f"  Fraud %: {y_bal.mean()*100:.1f}%")

    return X_bal, y_bal



## normuojam


In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled   = scaler.transform(X_val)
X_test_scaled  = scaler.transform(X_test)

lbfgs:

* Default solveris
* Tinka mažiems ir vidutiniams datasetams
* Stabilus, geras tikslumas
* Nėra geriausias labai dideliems duomenims

sag (Stochastic Average Gradient):

* Greitesnis dideliems datasetams (daug eilučių)
* Reikalauja standartizuotų duomenų
* Palaiko tik L2 regularizaciją
* Geras kai datasetas didelis ir paprastas

saga:

* Patobulintas sag variantas
* Geriausias pasirinkimas dideliems datasetams
* Palaiko L1, L2 ir elasticnet
* Tinka ir sparse (retoms) matricoms
* Reikalauja standartizacijos

L1 regularizacija:

* „bausmė“ už didelius koeficientus (naudoja |w|)
* Linkusi kai kuriuos svorius padaryti lygiai 0
* Automatiškai atlieka feature selection
* Naudinga, kai daug nereikalingų feature’ų
* L1 → „išmeta“ nereikalingus feature’us (nulina svorius)

L2 regularizacija:

* „bausmė“ už didelius koeficientus (naudoja w²)
* Sumažina svorius, bet retai padaro juos 0
* Modelis tampa stabilesnis (mažiau overfitting)
* Dažniausiai naudojamas default variantas
* L2 → „suspaudžia“ visus svorius, bet neišmeta

Kada kurį naudoti:

* mažas / vidutinis datasetas → lbfgs
* didelis datasetas → saga (dažniausiai geriausias)
* didelis datasetas + tik L2 → gali naudoti sag

## eks 1: nebalansuota

In [ ]:
print(f"Fraud procentas train aibėje: {y_train.mean()*100:.3f}%")

lr_no_bal = LogisticRegression(
    max_iter=1000,
    random_state=67,
    solver='saga',
    n_jobs=-1
)

cv_score(lr_no_bal, X_train_scaled, y_train, cv=5, label='Nebalansuota')

lr_no_bal.fit(X_train_scaled, y_train)
results_no_bal_val  = full_evaluate("LR – Nebalansuota", lr_no_bal, X_val_scaled,  y_val, threshold=0.1,  label='Validacija')
results_no_bal_test = full_evaluate("LR – Nebalansuota", lr_no_bal, X_test_scaled, y_test, threshold=0.1, label='Testas')

In [ ]:
import numpy as np
import pandas as pd

y_proba_val, _ = get_predictions(lr_no_bal, X_val_scaled)

results_no_bal = []

thresholds = np.arange(0.001, 0.55, 0.05)

for t in thresholds:
    y_pred = (y_proba_val >= t).astype(int)
    results_no_bal.append({
        'threshold': round(t, 3),
        'precision': precision_score(y_val, y_pred, zero_division=0),
        'recall': recall_score(y_val, y_pred, zero_division=0),
        'f1': f1_score(y_val, y_pred, zero_division=0),
        'pr_auc': average_precision_score(y_val, y_proba_val),
    })

df = pd.DataFrame(results_no_bal)
print(df.to_string(index=False))

print(df.nlargest(3, 'f1').to_string(index=False))

## eks 2: balansuota iki 10% fraud

In [ ]:
X_train_10, y_train_10 = balance_data(X_train_scaled, y_train, fraud_target_ratio=0.10)

lr_10 = LogisticRegression(
    max_iter=1000,
    random_state=67,
    solver='lbfgs',
    n_jobs=-1
)

cv_score(lr_10, X_train_10, y_train_10, cv=5, label='10% fraud')

lr_10.fit(X_train_10, y_train_10)
results_10_val  = full_evaluate("LR – 10% fraud", lr_10, X_val_scaled,  y_val,   label='Validacija')
results_10_test = full_evaluate("LR – 10% fraud", lr_10, X_test_scaled, y_test, label='Testas')

In [ ]:
import numpy as np
import pandas as pd

y_proba_val, _ = get_predictions(lr_10, X_val_scaled)

results_10 = []

thresholds = np.arange(0.001, 0.55, 0.05)

for t in thresholds:
    y_pred = (y_proba_val >= t).astype(int)
    results_10.append({
        'threshold': round(t, 3),
        'precision': precision_score(y_val, y_pred, zero_division=0),
        'recall': recall_score(y_val, y_pred, zero_division=0),
        'f1': f1_score(y_val, y_pred, zero_division=0),
        'pr_auc': average_precision_score(y_val, y_proba_val),
    })

df = pd.DataFrame(results_10)
print(df.to_string(index=False))

print(df.nlargest(3, 'f1').to_string(index=False))

## eks 2: balansuota iki 33% fraud

In [ ]:
X_train_33, y_train_33 = balance_data(X_train_scaled, y_train, fraud_target_ratio=0.33)

lr_33 = LogisticRegression(
    max_iter=1000,
    random_state=67,
    solver='saga',
    n_jobs=-1
)

cv_score(lr_33, X_train_33, y_train_33, cv=5, label='33% fraud')

lr_33.fit(X_train_33, y_train_33)
results_33_val  = full_evaluate("LR – 33% fraud", lr_33, X_val_scaled,  y_val,  label='Validacija')
results_33_test = full_evaluate("LR – 33% fraud", lr_33, X_test_scaled, y_test, label='Testas')

In [ ]:
import numpy as np
import pandas as pd

y_proba_val, _ = get_predictions(lr_33, X_val_scaled)

results_33 = []

thresholds = np.arange(0.001, 0.55, 0.05)

for t in thresholds:
    y_pred = (y_proba_val >= t).astype(int)
    results_33.append({
        'threshold': round(t, 3),
        'precision': precision_score(y_val, y_pred, zero_division=0),
        'recall': recall_score(y_val, y_pred, zero_division=0),
        'f1': f1_score(y_val, y_pred, zero_division=0),
        'pr_auc': average_precision_score(y_val, y_proba_val),
    })

df = pd.DataFrame(results_33)
print(df.to_string(index=False))

print(df.nlargest(3, 'f1').to_string(index=False))

## class_weight='balanced'

In [ ]:
lr_cw = LogisticRegression(
    class_weight='balanced',
    max_iter=1000,
    random_state=67,
    solver='saga',
    n_jobs=-1
)

cv_score(lr_cw, X_train_scaled, y_train, cv=5, label='class_weight=balanced')

lr_cw.fit(X_train_scaled, y_train)
results_cw_val  = full_evaluate("LR – class_weight balanced", lr_cw, X_val_scaled,  y_val,  label='Validacija')
results_cw_test = full_evaluate("LR – class_weight balanced", lr_cw, X_test_scaled, y_test, label='Testas')

## Geriausio modelio išsaugojimas

In [ ]:
import joblib, json, os
import numpy as np
from sklearn.metrics import f1_score, average_precision_score

os.makedirs('models', exist_ok=True)

def find_best_threshold(model, X_val, y_val, metric='f1'):
    y_proba = model.predict_proba(X_val)[:, 1]
    thresholds = np.arange(0.001, 0.80, 0.005)
    best_t, best_score = 0.5, -1
    for t in thresholds:
        y_pred = (y_proba >= t).astype(int)
        score = f1_score(y_val, y_pred, zero_division=0)
        if score > best_score:
            best_score = score
            best_t = t
    pr_auc = average_precision_score(y_val, y_proba)
    return best_t, best_score, pr_auc

experiments = {
    'lr_no_bal': lr_no_bal,
    'lr_10': lr_10,
    'lr_33': lr_33,
    'lr_cw': lr_cw,
}
summary = {}

for name, model in experiments.items():
    best_t, best_f1, pr_auc = find_best_threshold(model, X_val_scaled, y_val)
    
    joblib.dump(model, f'models/{name}.pkl')
    
    summary[name] = {
        'threshold': round(best_t, 4),
        'val_f1': round(best_f1, 4),
        'val_pr_auc': round(pr_auc, 4),
        'model_file': f'{name}.pkl',
    }
    print(f"{name:15s} | threshold={best_t:.3f} | F1={best_f1:.4f} | PR-AUC={pr_auc:.4f}")

best_name = max(summary, key=lambda k: summary[k]['val_pr_auc'])
best_info = summary[best_name]

print(f"Geriausias modelis: {best_name} (PR-AUC={best_info['val_pr_auc']})")

joblib.dump(experiments[best_name], 'models/best_model.pkl')
joblib.dump(scaler, 'models/scaler.pkl')

meta = {
    'best_model': best_name,
    'threshold': best_info['threshold'],
    'val_f1': best_info['val_f1'],
    'val_pr_auc': best_info['val_pr_auc'],
    'all_experiments': summary,
}

with open('models/meta.json', 'w') as f:
    json.dump(meta, f, indent=2)
